In [2]:
# Cell 1: Imports
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind, chi2_contingency, f_oneway
import matplotlib.pyplot as plt
import seaborn as sns

# Set styling for plots
sns.set(style="whitegrid")


In [6]:
# Cell 2: Load Dataset
# Replace with your actual path
df = pd.read_csv("../dataset/MachineLearningRating_v3.txt", sep="|", engine='python')

# Preview
df.head()


,UnderwrittenCoverID,PolicyID,TransactionMonth,IsVATRegistered,Citizenship,LegalType,Title,Language,Bank,AccountType,...,ExcessSelected,CoverCategory,CoverType,CoverGroup,Section,Product,StatutoryClass,StatutoryRiskType,TotalPremium,TotalClaims
0,145249,12827,2015-03-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,21.929825,0.0
1,145249,12827,2015-05-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,21.929825,0.0
2,145249,12827,2015-07-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,0.000000,0.0
3,145255,12827,2015-05-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Metered Taxis - R2000,Own damage,Own Damage,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,512.848070,0.0
4,145255,12827,2015-07-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Metered Taxis - R2000,Own damage,Own Damage,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,0.000000,0.0


In [7]:
# Cell 3: Create Metrics for Analysis
df['Margin'] = df['TotalPremium'] - df['TotalClaims']
df['HasClaim'] = df['TotalClaims'] > 0
df['ClaimSeverity'] = df['TotalClaims'].where(df['TotalClaims'] > 0, np.nan)

df[['TotalPremium', 'TotalClaims', 'Margin', 'HasClaim', 'ClaimSeverity']].describe()


,TotalPremium,TotalClaims,Margin,ClaimSeverity
count,1.000098e+06,1.000098e+06,1.000098e+06,2788.000000
mean,6.190550e+01,6.486119e+01,-2.955694e+00,23273.387063
std,2.302845e+02,2.384075e+03,2.367137e+03,38719.512597
min,-7.825768e+02,-1.200241e+04,-3.928486e+05,139.043860
25%,0.000000e+00,0.000000e+00,0.000000e+00,1680.728070
50%,2.178333e+00,0.000000e+00,2.157687e+00,6140.350877
75%,2.192982e+01,0.000000e+00,2.192982e+01,30480.991228
max,6.528260e+04,3.930921e+05,6.528260e+04,393092.105263


In [8]:
# Cell 4: H₀: No Risk Difference Across Provinces

print("🔍 H₀: No risk differences across provinces")

# 1. Frequency - Chi2
province_freq = pd.crosstab(df['Province'], df['HasClaim'])
chi2, p_freq, _, _ = chi2_contingency(province_freq)

# 2. Severity - ANOVA
province_sev_groups = [group.dropna() for _, group in df.groupby('Province')['ClaimSeverity']]
fstat, p_sev = f_oneway(*province_sev_groups)

print(f"✅ Claim Frequency p-value: {p_freq:.4f}")
print(f"✅ Claim Severity p-value: {p_sev:.4f}")

print("👉 Frequency:", "Reject H₀" if p_freq < 0.05 else "Fail to reject H₀")
print("👉 Severity:", "Reject H₀" if p_sev < 0.05 else "Fail to reject H₀")


🔍 H₀: No risk differences across provinces
✅ Claim Frequency p-value: 0.0000
✅ Claim Severity p-value: 0.0000
👉 Frequency: Reject H₀
👉 Severity: Reject H₀


### There are statistically significant differences in both claim frequency and severity across provinces. This suggests that regional risk segmentation is necessary.

In [9]:
# Cell 5: H₀: No Risk Difference Between Zip Codes (Top 5)

print("🔍 H₀: No risk differences between top zip codes")

# Select top zip codes
top_zip = df['PostalCode'].value_counts().nlargest(5).index
df_zip = df[df['PostalCode'].isin(top_zip)]

# Frequency - Chi2
zip_freq = pd.crosstab(df_zip['PostalCode'], df_zip['HasClaim'])
chi2_zip, p_zip_freq, _, _ = chi2_contingency(zip_freq)

# Severity - ANOVA
zip_sev_groups = [group.dropna() for _, group in df_zip.groupby('PostalCode')['ClaimSeverity']]
fstat_zip, p_zip_sev = f_oneway(*zip_sev_groups)

print(f"✅ Claim Frequency p-value: {p_zip_freq:.4f}")
print(f"✅ Claim Severity p-value: {p_zip_sev:.4f}")

print("👉 Frequency:", "Reject H₀" if p_zip_freq < 0.05 else "Fail to reject H₀")
print("👉 Severity:", "Reject H₀" if p_zip_sev < 0.05 else "Fail to reject H₀")


🔍 H₀: No risk differences between top zip codes
✅ Claim Frequency p-value: 0.0000
✅ Claim Severity p-value: 0.0039
👉 Frequency: Reject H₀
👉 Severity: Reject H₀


### Certain zip codes show significantly higher claim rates and claim sizes. This may reflect local driving conditions or crime rates.

In [10]:
# Cell 7: H₀: No Risk Difference Between Men and Women

print("🔍 H₀: No risk difference between Women and Men")

# Frequency - Chi2
gender_freq = pd.crosstab(df['Gender'], df['HasClaim'])
chi2_gender, p_gender_freq, _, _ = chi2_contingency(gender_freq)

# Severity - T-test
male_claims = df[df['Gender'] == 'Male']['ClaimSeverity'].dropna()
female_claims = df[df['Gender'] == 'Female']['ClaimSeverity'].dropna()
stat_gender, p_gender_sev = ttest_ind(male_claims, female_claims, equal_var=False)

print(f"✅ Claim Frequency p-value: {p_gender_freq:.4f}")
print(f"✅ Claim Severity p-value: {p_gender_sev:.4f}")

print("👉 Frequency:", "Reject H₀" if p_gender_freq < 0.05 else "Fail to reject H₀")
print("👉 Severity:", "Reject H₀" if p_gender_sev < 0.05 else "Fail to reject H₀")


🔍 H₀: No risk difference between Women and Men
✅ Claim Frequency p-value: 0.0266
✅ Claim Severity p-value: 0.5680
👉 Frequency: Reject H₀
👉 Severity: Fail to reject H₀


### Women file fewer claims than men, but the average size of those claims is similar. This supports:

### Potential discounted premiums for female drivers.

### Marketing campaigns targeted at female low-risk profiles.